# Train Qwen for price tool calls



In [16]:
# Run once if these libraries are missing.
# %pip install -U torch transformers datasets peft accelerate matplotlib

import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from datasets import Dataset, DatasetDict
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

In [17]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "jsonl_files" / "dataset_normal.jsonl").exists():
    PROJECT_DIR = PROJECT_DIR / "price_retrieval"

DATASET_PATH = PROJECT_DIR / "jsonl_files" / "dataset_normal.jsonl"
ADAPTER_PATH = PROJECT_DIR / "models" / "qwen3_price_lora"
MERGED_PATH = PROJECT_DIR / "models" / "qwen3_price_merged"
MODEL_NAME = "Qwen/Qwen3-0.6B"

MAX_LENGTH = 512       # Maximum tokens kept for one training example.
TEST_SIZE = 0.10
SEED = 42
EPOCHS = 3
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
LEARNING_RATE = 1e-4

SYSTEM_PROMPT = (
    "Convert the user's stock-price request into one get_prices function call. "
    "Return only the function call with valid arguments. Do not answer the request."
)

config = json.loads((PROJECT_DIR / "config.json").read_text())
HF_TOKEN = config.get("hg_access_token")
CACHE_DIR = config.get("cache_dir")

In [18]:
if torch.cuda.is_available():
    DEVICE = "cuda"
    USE_BF16 = torch.cuda.is_bf16_supported()
    MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = "xpu"
    USE_BF16 = True
    MODEL_DTYPE = torch.bfloat16
else:
    DEVICE = "cpu"
    USE_BF16 = False
    MODEL_DTYPE = torch.float32

print("Training device:", DEVICE)
print("Model dtype:", MODEL_DTYPE)

Training device: cpu
Model dtype: torch.float32


In [19]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

with DATASET_PATH.open() as file:
    rows = [json.loads(line) for line in file if line.strip()]

print("Loaded rows:", len(rows))

Loaded rows: 2629


In [20]:
# Keep paraphrases from the same iteration in the same split.
group_ids = list({row["metadata"]["iteration"] for row in rows})
random.Random(SEED).shuffle(group_ids)
test_count = max(1, round(len(group_ids) * TEST_SIZE))
test_groups = set(group_ids[:test_count])

train_rows = [row for row in rows if row["metadata"]["iteration"] not in test_groups]
test_rows = [row for row in rows if row["metadata"]["iteration"] in test_groups]

print("Train rows:", len(train_rows))
print("Test rows:", len(test_rows))

Train rows: 2369
Test rows: 260


In [22]:
def make_text(row):
    system = {"role": "system", "content": SYSTEM_PROMPT}
    messages = [system] + row["messages"]

    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"prompt_text": prompt_text, "full_text": full_text}


train_dataset = Dataset.from_list([make_text(row) for row in train_rows])
test_dataset = Dataset.from_list([make_text(row) for row in test_rows])
splits = DatasetDict({"train": train_dataset, "test": test_dataset})

print(splits)
print(splits["train"][0]["full_text"])

DatasetDict({
    train: Dataset({
        features: ['prompt_text', 'full_text'],
        num_rows: 2369
    })
    test: Dataset({
        features: ['prompt_text', 'full_text'],
        num_rows: 260
    })
})
<|im_start|>system
Convert the user's stock-price request into one get_prices function call. Return only the function call with valid arguments. Do not answer the request.<|im_end|>
<|im_start|>user
Can you provide the daily prices for Intuitive Surgical, American International Group, and I from April 2020 to March 2023?<|im_end|>
<|im_start|>assistant
<think>

</think>

<tool_call>
{"name": "get_prices", "arguments": {"queries": [{"symbols": ["ISRG"], "timeframe": ["daily"], "start_month": "April", "start_year": 2020, "end_month": "March", "end_year": 2023}, {"symbols": ["AIG"], "timeframe": ["daily"], "start_month": "April", "start_year": 2020, "end_month": "March", "end_year": 2023}]}}
</tool_call><|im_end|>



In [24]:
OUTPUT_PATH = PROJECT_DIR / "dataset" / "huggingface" / "qwen3_price_tool_calling"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
splits.save_to_disk(str(OUTPUT_PATH))

print("Dataset saved to:", OUTPUT_PATH)
print(splits)

Saving the dataset (1/1 shards): 100%|██████████| 260/260 [00:00<00:00, 121114.95 examples/s]

Dataset saved to: /home/erfan/agentic_finance/price_retrieval/dataset/huggingface/qwen3_price_tool_calling
DatasetDict({
    train: Dataset({
        features: ['prompt_text', 'full_text'],
        num_rows: 2369
    })
    test: Dataset({
        features: ['prompt_text', 'full_text'],
        num_rows: 260
    })
})


In [23]:
def tokenize_row(row):
    full = tokenizer(
        row["full_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )
    prompt = tokenizer(
        row["prompt_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )

    labels = full["input_ids"].copy()
    prompt_length = min(len(prompt["input_ids"]), len(labels))
    labels[:prompt_length] = [-100] * prompt_length

    return {
        "input_ids": full["input_ids"],
        "attention_mask": full["attention_mask"],
        "labels": labels,
    }


tokenized = splits.map(
    tokenize_row,
    remove_columns=["prompt_text", "full_text"],
)

target_lengths = [
    sum(label != -100 for label in labels)
    for labels in tokenized["train"]["labels"]
]
print("Shortest assistant target:", min(target_lengths), "tokens")
print("Longest example:", max(map(len, tokenized["train"]["input_ids"])), "tokens")

Map: 100%|██████████| 260/260 [00:00<00:00, 3032.31 examples/s]


Shortest assistant target: 73 tokens
Longest example: 256 tokens


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
    torch_dtype=MODEL_DTYPE,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 976.72it/s] 


trainable params: 1,146,880 || all params: 597,196,800 || trainable%: 0.1920


In [27]:
training_args = TrainingArguments(
    output_dir=str(ADAPTER_PATH),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=40,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    bf16=USE_BF16,
    fp16=DEVICE == "cuda" and not USE_BF16,
    report_to="none",
    load_best_model_at_end=True,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
result = trainer.train()
trainer.save_model(str(ADAPTER_PATH))
tokenizer.save_pretrained(str(ADAPTER_PATH))
result.metrics

In [ ]:
train_loss = [(row["step"], row["loss"]) for row in trainer.state.log_history if "loss" in row]
eval_loss = [(row["step"], row["eval_loss"]) for row in trainer.state.log_history if "eval_loss" in row]

if train_loss:
    plt.plot(*zip(*train_loss), label="train loss")
if eval_loss:
    plt.plot(*zip(*eval_loss), label="evaluation loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Optional: merge before converting to GGUF

Run the next cell after training. The merged model can then be converted with llama.cpp's `convert_hf_to_gguf.py` and quantized with `llama-quantize`.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16,
)
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH).merge_and_unload()
merged_model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)
print("Merged model saved to:", MERGED_PATH)